General program

In [12]:
# Cell 1: Import dependencies and define Core Engine with Fault Analysis Tools
import numpy as np
import re
import os
import time
import json

class DiagnosticTensorSimulator:
    def __init__(self):
        self.inputs = []
        self.outputs = []
        self.gates = {}       
        self.topo_order = []  

    def load_bench(self, file_path: str):
        self.inputs = []
        self.outputs = []
        self.gates = {}
        
        if not os.path.exists(file_path):
            raise FileNotFoundError(f"🚨 Path '{file_path}' does not exist.")
            
        with open(file_path, 'r') as f:
            lines = [line.strip() for line in f if line.strip() and not line.startswith('#')]
        
        for line in lines:
            if line.startswith("INPUT"):
                node = re.search(r'INPUT\s*\(\s*([\w\d_-]+)\s*\)', line).group(1)
                self.inputs.append(node)
            elif line.startswith("OUTPUT"):
                node = re.search(r'OUTPUT\s*\(\s*([\w\d_-]+)\s*\)', line).group(1)
                self.outputs.append(node)
            elif "=" in line:
                out_node, expr = line.split("=")
                out_node = out_node.strip()
                gate_type, ins_raw = expr.split("(")
                gate_type = gate_type.strip().upper()
                ins = [i.strip() for i in ins_raw.replace(")", "").split(",")]
                self.gates[out_node] = (gate_type, ins)

        # Topological Sort
        all_resolved = set(self.inputs)
        remaining_gates = dict(self.gates)
        self.topo_order = []
        
        while remaining_gates:
            ready_gates = [g for g, (gt, ins) in remaining_gates.items() if all(i in all_resolved for i in ins)]
            if not ready_gates:
                raise ValueError("🚨 Feedback loop or missing dependency discovered!")
            for g in ready_gates:
                self.topo_order.append(g)
                all_resolved.add(g)
                del remaining_gates[g]

    def get_collapsed_fault_list(self):
        """
        Generates a true local equivalence-collapsed stuck-at fault list.
        Applies standard gate collapsing rules (e.g., for NAND, input SA0s collapse to output SA1).
        """
        # Start by tracking all faults we potentially want to look at
        # We use a set to automatically handle shared nets/wires smoothly
        collapsed_faults = set()
        
        # 1. Add all faults for Primary Inputs as a baseline baseline
        for pi in self.inputs:
            collapsed_faults.add((pi, "SA0"))
            collapsed_faults.add((pi, "SA1"))
            
        # 2. Iterate through every gate and apply Boolean Equivalence rules
        for gate_out, (gate_type, gate_ins) in self.gates.items():
            # Always track both faults on the gate output to cover downstream paths
            collapsed_faults.add((gate_out, "SA0"))
            collapsed_faults.add((gate_out, "SA1"))
            
            if gate_type in ["NAND", "AND"]:
                # Rule: Input SA0 faults are equivalent to Output SA0 (for AND) or Output SA1 (for NAND)
                # Therefore, we can REMOVE/COLLAPSE all input SA0 faults from our tracking list!
                for inp in gate_ins:
                    # Keep input SA1 (it is distinct), but discard input SA0
                    collapsed_faults.add((inp, "SA1"))
                    if (inp, "SA0") in collapsed_faults and inp not in self.inputs:
                        collapsed_faults.remove((inp, "SA0"))
                        
            elif gate_type in ["NOR", "OR"]:
                # Rule: Input SA1 faults are equivalent to Output SA1 (for OR) or Output SA0 (for NOR)
                # Therefore, we can REMOVE/COLLAPSE all input SA1 faults!
                for inp in gate_ins:
                    # Keep input SA0, discard input SA1
                    collapsed_faults.add((inp, "SA0"))
                    if (inp, "SA1") in collapsed_faults and inp not in self.inputs:
                        collapsed_faults.remove((inp, "SA1"))
                        
            elif gate_type == "NOT":
                # Rule: Input SA0 == Output SA1, Input SA1 == Output SA0
                # We can completely drop internal NOT inputs if they aren't PIs
                for inp in gate_ins:
                    if inp not in self.inputs:
                        collapsed_faults.discard((inp, "SA0"))
                        collapsed_faults.discard((inp, "SA1"))
            else:
                # Fallback for complex gates (XOR/XNOR) where static local collapsing doesn't apply cleanly
                for inp in gate_ins:
                    collapsed_faults.add((inp, "SA0"))
                    collapsed_faults.add((inp, "SA1"))

        # Convert back to a clean, unique sorted list of tuples
        return sorted(list(collapsed_faults))

    def simulate(self, input_vectors: np.ndarray, fault_node=None, fault_type=None) -> dict:
        num_vectors = input_vectors.shape[1]
        node_values = {}
        
        for idx, inp in enumerate(self.inputs):
            node_values[inp] = input_vectors[idx].copy()
            if fault_node == inp:
                node_values[inp] = np.zeros(num_vectors, dtype=np.uint8) if fault_type == 'SA0' else np.ones(num_vectors, dtype=np.uint8)

        for gate in self.topo_order:
            gate_type, ins = self.gates[gate]
            in_vals = [node_values[i] for i in ins]
            
            if gate_type == "NAND":
                res = ~(np.bitwise_and.reduce(in_vals)) & 1
            elif gate_type == "AND":
                res = np.bitwise_and.reduce(in_vals)
            elif gate_type == "OR":
                res = np.bitwise_or.reduce(in_vals)
            elif gate_type == "NOR":
                res = ~(np.bitwise_or.reduce(in_vals)) & 1
            elif gate_type == "XOR":
                res = np.bitwise_xor.reduce(in_vals)
            elif gate_type == "XNOR":
                res = ~(np.bitwise_xor.reduce(in_vals)) & 1
            elif gate_type == "NOT":
                res = ~in_vals[0] & 1
            elif gate_type in ["BUFF", "BUF"]:
                res = in_vals[0].copy()
            else:
                raise NotImplementedError(f"Gate type '{gate_type}' is unhandled.")
                
            if fault_node == gate:
                res = np.zeros(num_vectors, dtype=np.uint8) if fault_type == 'SA0' else np.ones(num_vectors, dtype=np.uint8)
                    
            node_values[gate] = res
            
        return {out: node_values[out] for out in self.outputs}

In [13]:
# Cell 2: Test Vector Utility Loader
def load_test_vectors(file_path: str, num_inputs: int, fallback_count=100) -> np.ndarray:
    if file_path and os.path.exists(file_path):
        vectors = []
        with open(file_path, 'r') as f:
            for line in f:
                line = line.strip()
                if not line or line.startswith('#'):
                    continue
                binary_match = re.match(r'^([01]+)', line)
                if binary_match:
                    pi_bits = binary_match.group(1)
                    if len(pi_bits) != num_inputs:
                        raise ValueError(f"🚨 Vector bit length mismatch! Circuit expects {num_inputs} inputs.")
                    vectors.append([int(b) for b in pi_bits])
        if vectors:
            print(f"📖 Successfully loaded {len(vectors)} test vectors from '{file_path}'.")
            return np.array(vectors, dtype=np.uint8).T
            
    print(f"⚠️ File '{file_path}' not found. Generating {fallback_count} random patterns dynamically...")
    np.random.seed(42)
    return np.random.randint(0, 2, size=(num_inputs, fallback_count), dtype=np.uint8)

In [14]:
# Cell 3: Automated Diagnostic Batch Testing & Log Generator
def run_automated_fault_assessment(circuit, input_matrix, log_filename="fault_simulation_diagnostics.log"):
    """
    Executes all collapsed faults, benchmarks evaluation time, 
    gathers advanced diagnostic metrics, and streams results to a structured log file.
    """
    # 1. Acquire Golden Reference Output
    golden_outputs = circuit.simulate(input_matrix)
    total_vectors = input_matrix.shape[1]
    
    # 2. Get unique checkpoint faults
    fault_list = circuit.get_collapsed_fault_list()
    print(f"🔬 Total unique collapsed faults to analyze: {len(fault_list)}")
    
    log_entries = []
    detected_faults_count = 0
    
    print(f"✍️ Simulating and writing records directly to: '{log_filename}'...")
    bench_start=time.perf_counter()
    with open(log_filename, "w") as log_file:
        # Write header line
        log_file.write("# === VLSI TENSOR SIMULATION DIAGNOSTIC REPORT ===\n")
        log_file.write(f"# Timestamp: {time.strftime('%Y-%m-%d %H:%M:%S')}\n")
        log_file.write(f"# Total Test Vectors Evaluated: {total_vectors}\n")
        log_file.write(f"# Total PIs count: {len(circuit.inputs)},list of PIs:{circuit.inputs}\n")
        log_file.write(f"# Total POs count: {len(circuit.outputs)},list of POS: {circuit.outputs}\n")
        log_file.write(f"# Total fault list count: {len(fault_list)},list of fault list: {fault_list}\n\n")
        
        for f_node, f_type in fault_list:
            # Benchmark precisely using high-resolution performance timers
            start_time = time.perf_counter()
            faulty_outputs = circuit.simulate(input_matrix, fault_node=f_node, fault_type=f_type)
            end_time = time.perf_counter()
            
            duration_microseconds = (end_time - start_time) * 1_000_000
            
            # Metric Discovery: Compute detection vectors
            detection_mask = np.zeros(total_vectors, dtype=bool)
            corrupted_pos = []
            hamming_distance_per_po = {}
            
            for out in circuit.outputs:
                mismatch_mask = (golden_outputs[out] != faulty_outputs[out])
                detection_mask |= mismatch_mask
                
                mismatch_count = int(np.sum(mismatch_mask))
                if mismatch_count > 0:
                    corrupted_pos.append(out)
                    hamming_distance_per_po[out] = mismatch_count
            
            is_detected = bool(np.any(detection_mask))
            detect_indices = np.where(detection_mask)[0].tolist()
            
            if is_detected:
                detected_faults_count += 1
                
            # Build an enriched diagnostic payload record
            diagnostic_record = {
                "fault_node": f_node,
                "fault_type": f_type,
                "status": "DETECTED" if is_detected else "UNDETECTED",
                "duration_us": round(duration_microseconds, 8),
                "total_detecting_vectors": len(detect_indices),
                "detecting_vector_indices": detect_indices,
                "corrupted_primary_outputs": corrupted_pos,
                "output_mismatch_counts": hamming_distance_per_po, #sum of all results where golder po <> fault po
                "raw_golden_po": {out: golden_outputs[out].tolist() for out in circuit.outputs},
                "raw_faulty_po": {out: faulty_outputs[out].tolist() for out in circuit.outputs}
            }
            
            # Write row directly to disk as JSON string lines for easy streaming parsing later
            log_file.write(json.dumps(diagnostic_record) + "\n")
            log_entries.append(diagnostic_record)
        bench_end=time.perf_counter()
        duration_bench_us = (bench_end - bench_start) * 1_000_000
        log_file.write(f"{'*'*100}\nCoverage: {(detected_faults_count / len(fault_list)) * 100}%\n")
        log_file.write(f"{'*'*100}\nTotal time: {duration_bench_us}")
    # Calculate Summary Stats
    coverage = (detected_faults_count / len(fault_list)) * 100
    print(f"🎉 Analysis Complete! Overall Fault Coverage: {coverage:.2f}%")
    return log_entries

In [15]:
# Cell 4: Production Run
# BENCH_PATH = "./data.nogit/c17.bench"
# VECTOR_PATH = "./data.nogit/c17.tests"
# LOG_PATH = "./logs/c17_diagnostics_report.log"
def benchmark(bench,test,log):
    # Initialize and run
    circuit = DiagnosticTensorSimulator()
    circuit.load_bench(bench)

    input_matrix = load_test_vectors(test, num_inputs=len(circuit.inputs), fallback_count=100)

    # Run full evaluation
    records = run_automated_fault_assessment(circuit, input_matrix, log_filename=log)

    # # Preview the first log record inside your notebook display
    # print("\n👀 Sample JSON Log Entry Preview (First Fault Tested):")
    # print(json.dumps(records[0], indent=4))

In [7]:
# Cell 4: Production Run
BENCH_PATH = "./data.nogit/c6288.bench"
VECTOR_PATH = "./data.nogit/c6288.tests"
LOG_PATH = "./logs/c6288_diagnostics_report.log"
benchmark(BENCH_PATH,VECTOR_PATH,LOG_PATH)

📖 Successfully loaded 100 test vectors from './data.nogit/c6288.tests'.


In [ ]:
import os
bench_file=[]
test_file=[]
log_file=[]
for a in os.listdir('./data.nogit/'):
    if ".bench" in a:
        bench_file.append('./data.nogit/'+a)
        log_file.append('./logs/'+a.split('.')[0]+'_report.log')
    else:
        test_file.append('./data.nogit/'+a)
bench_file.sort()
test_file.sort()
log_file.sort()

for a,b,c in zip(bench_file,test_file,log_file):
    print(a,'\t',b,'\t',c)
    try:
        benchmark(a,b,c)
    except:
        print(f"{a} failed, pending to run benchmark again")

./data.nogit/c1355.bench 	 ./data.nogit/c1355.tests 	 ./logs/c1355_report.log
📖 Successfully loaded 100 test vectors from './data.nogit/c1355.tests'.
🔬 Total unique collapsed faults to analyze: 692
✍️ Simulating and writing records directly to: './logs/c1355_report.log'...
./data.nogit/c1355.bench failed, pending to run benchmark again
./data.nogit/c17.bench 	 ./data.nogit/c17.tests 	 ./logs/c17_report.log
📖 Successfully loaded 100 test vectors from './data.nogit/c17.tests'.
🔬 Total unique collapsed faults to analyze: 18
✍️ Simulating and writing records directly to: './logs/c17_report.log'...
🎉 Analysis Complete! Overall Fault Coverage: 100.00%
./data.nogit/c1908.bench 	 ./data.nogit/c1908.tests 	 ./logs/c1908_report.log
📖 Successfully loaded 100 test vectors from './data.nogit/c1908.tests'.
🔬 Total unique collapsed faults to analyze: 948
✍️ Simulating and writing records directly to: './logs/c1908_report.log'...
🎉 Analysis Complete! Overall Fault Coverage: 82.49%
./data.nogit/c2670.b

In [ ]:
BENCH_PATH = "./data.nogit/c17.bench"
VECTOR_PATH = "./data.nogit/c17.tests"
LOG_PATH = "./logs/c17_diagnostics_report.log"
benchmark(BENCH_PATH,VECTOR_PATH,LOG_PATH)

📖 Successfully loaded 100 test vectors from './data.nogit/c17.tests'.
🔬 Total unique collapsed faults to analyze: 18
✍️ Simulating and writing records directly to: './logs/c17_diagnostics_report.log'...
🎉 Analysis Complete! Overall Fault Coverage: 100.00%


In [1]:
import json
import os
import re

def generate_summary_from_header(log_filepath, bench_name="Circuit"):
    """
    Parses metadata directly from the structured log file header,
    then aggregates runtime and fault coverage data from the log body.
    """
    if not os.path.exists(log_filepath):
        print(f"🚨 Log file '{log_filepath}' not found.")
        return None

    # Header tracking elements
    pi_count = 0
    po_count = 0
    fault_list_count = 0
    
    # Body calculations
    detected_faults = 0
    cumulative_time_us = 0.0

    with open(log_filepath, "r") as f:
        for line in f:
            line = line.strip()
            
            # 1. Parse Metadata Metrics out of the Header Comments
            if line.startswith("#"):
                if "Total PIs count:" in line:
                    match = re.search(r'Total PIs count:\s*(\d+)', line)
                    if match: pi_count = int(match.group(1))
                elif "Total POs count:" in line:
                    match = re.search(r'Total POs count:\s*(\d+)', line)
                    if match: po_count = int(match.group(1))
                elif "Total fault list count:" in line:
                    match = re.search(r'Total fault list count:\s*(\d+)', line)
                    if match: fault_list_count = int(match.group(1))
                continue
                
            # 2. Parse Execution Records from the Data Rows
            if not line:
                continue
                
            try:
                record = json.loads(line)
                cumulative_time_us += record.get("duration_us", 0.0)
                if record.get("status") == "DETECTED":
                    detected_faults += 1
            except json.JSONDecodeError:
                continue

    # Final Metric Balancing Calculations
    fault_coverage = (detected_faults / fault_list_count * 100) if fault_list_count > 0 else 0.0
    total_time_ms = cumulative_time_us / 1000.0
    properties_str = f"{pi_count} PIs / {po_count} POs"
    
    # Build cleanly formatted ASCII Output Block
    header_str = f"| {'Circuit':<12} | {'Properties':<16} | {'Fault List Count':<18} | {'Total Time':<14} | {'Fault Coverage':<14} |"
    divider = "+" + "-"*14 + "+" + "-"*18 + "+" + "-"*20 + "+" + "-"*16 + "+" + "-"*16 + "+"
    #row_str = f"| {bench_name:<12} | {properties_str:<16} | {fault_list_count:<18} | {total_time_ms:.2f} ms:<14}| {fault_coverage:.2f}%:<14} |"
    
    # Correct string slicing for uniform row cells
    row_str = f"| {bench_name:<12} | {properties_str:<16} | {fault_list_count:<18} | {f'{total_time_ms:.2f} ms':<14} | {f'{fault_coverage:.2f}%':<14} |"

    print("\n📊 === VLSI FAULT SIMULATION BASELINE SUMMARY ===")
    print(divider)
    print(header_str)
    print(divider)
    print(row_str)
    print(divider)

In [ ]:
import re
import time
import torch

class BenchCircuitParser:
    def __init__(self, bench_file_path: str, device: str = None):
        """
        Initializes the VLSI Bench Parser and compiles structural gates into dense 
        Torch Tensors for high-throughput parallel CPU/GPU logic and fault simulations.
        
        :param bench_file_path: Path to the target .bench netlist file.
        :param device: 'cuda' for GPU acceleration, 'cpu' for standard tensor layout. 
                       Defaults to auto-detecting CUDA availability.
        """
        self.file_path = bench_file_path
        self.t0 = time.perf_counter()
        
        # Select device automatically if not specified
        if device is None:
            self.device = "cuda" if torch.cuda.is_available() else "cpu"
        else:
            self.device = device
            
        # Core structural attributes
        self.primary_inputs = []
        self.primary_outputs = []
        self.gates = {}             
        self.circuit = []           
        self.signals = {}           
        self.all_signals = []       
        self.collapsed_faults = []  
        
        # Numerical Indexing Maps for Tensor Compilation
        self.signal_to_idx = {}
        self.idx_to_signal = {}
        self.tensor_layers = []  # List of dicts containing compiled evaluation layers
        
        # Diagnostics
        self.fanout_stems = 0
        self.gate_inputs = 0
        
        # Pipeline execution
        self._parse_bench_file()
        self._insert_fanout_buffers()
        self._compute_topological_order()
        self._generate_signals_map()
        self._run_fault_equivalence()
        
        # High-Performance Tensor Matrix compilation stage
        self._compile_to_tensors()
        self._print_summary()

    def log(self, msg: str):
        print(f"# {time.perf_counter() - self.t0:09.3f} - {msg}")

    def _parse_line(self, line: str):
        line = line.strip()
        if "=" not in line:
            return None
        try:
            output_part, gate_part = line.split("=", 1)
            output_signal = output_part.strip()
            match = re.match(r"^\s*([A-Za-z0-9_]+)\s*\((.*)\)\s*$", gate_part.strip())
            if match:
                gate_type = match.group(1).upper()
                inputs = [i.strip() for i in match.group(2).split(",") if i.strip()]
                return output_signal, gate_type, inputs
        except Exception:
            pass
        return None

    def _parse_bench_file(self):
        self.log(f"Parsing file: {self.file_path}")
        with open(self.file_path, "r") as f:
            for line in f:
                line = line.strip()
                if not line or line.startswith("#"):
                    continue
                if line.startswith("INPUT("):
                    pi = line[6:line.index(")")].strip()
                    self.primary_inputs.append(pi)
                    continue
                if line.startswith("OUTPUT("):
                    po = line[7:line.index(")")].strip()
                    self.primary_outputs.append(po)
                    continue
                parsed = self._parse_line(line)
                if parsed:
                    out_sig, g_type, in_sigs = parsed
                    self.gates[out_sig] = {"type": g_type, "inputs": in_sigs}

    def _insert_fanout_buffers(self):
        consumption_counts = {}
        for out_sig, gate_info in list(self.gates.items()):
            for inp in gate_info["inputs"]:
                if inp not in consumption_counts:
                    consumption_counts[inp] = []
                consumption_counts[inp].append(out_sig)
        
        for src_signal, consumer_gates in consumption_counts.items():
            if len(consumer_gates) > 1:
                self.fanout_stems += 1
                for idx, consumer_gate in enumerate(consumer_gates):
                    buffer_out = f"{src_signal}.b{idx}"
                    self.gates[buffer_out] = {"type": "BUFF", "inputs": [src_signal]}
                    target_inputs = self.gates[consumer_gate]["inputs"]
                    for i_idx, inp in enumerate(target_inputs):
                        if inp == src_signal:
                            target_inputs[i_idx] = buffer_out

    def _compute_topological_order(self):
        in_degree = {gate: len(info["inputs"]) for gate, info in self.gates.items()}
        signal_to_gates = {}
        for gate, info in self.gates.items():
            for inp in info["inputs"]:
                if inp not in signal_to_gates:
                    signal_to_gates[inp] = []
                signal_to_gates[inp].append(gate)
                
        ready_queue = list(self.primary_inputs)
        ordered_gates = []
        
        # Track gate groups layer-by-layer for concurrent tensor executions
        self.layer_groups = []
        
        # Primary inputs form virtual baseline Layer 0
        current_layer = []
        next_layer = []
        
        while ready_queue or current_layer:
            if not current_layer:
                current_layer = list(ready_queue)
                ready_queue.clear()
                if not current_layer:
                    break
            
            curr = current_layer.pop(0)
            if curr in signal_to_gates:
                for dependent_gate in signal_to_gates[curr]:
                    in_degree[dependent_gate] -= 1
                    if in_degree[dependent_gate] == 0:
                        next_layer.append(dependent_gate)
                        ordered_gates.append(dependent_gate)
            
            if not current_layer and next_layer:
                self.layer_groups.append(list(next_layer))
                ready_queue.extend(next_layer)
                next_layer.clear()

        self.circuit = []
        for gate in ordered_gates:
            g_type = self.gates[gate]["type"]
            g_inps = ", ".join(self.gates[gate]["inputs"])
            self.circuit.append(f"{gate} = {g_type}({g_inps})")

    def _generate_signals_map(self):
        for gate_str in self.circuit:
            parsed = self._parse_line(gate_str)
            if parsed:
                out_sig, _, in_sigs = parsed
                for inp in in_sigs:
                    if inp not in self.signals:
                        self.signals[inp] = []
                    self.signals[inp].append(gate_str)
        self.all_signals = list(self.signals.keys())

    def _run_fault_equivalence(self):
        collapsed_set = set()
        equivalences = set()
        not_gates = []
        
        for gate_str in reversed(self.circuit):
            parsed = self._parse_line(gate_str)
            if not parsed: continue
            out_sig, gate_type, in_sigs = parsed
            
            if gate_type != "BUFF":
                self.gate_inputs += len(in_sigs)
            if out_sig in self.primary_outputs:
                collapsed_set.add(f"{out_sig}@0")
                collapsed_set.add(f"{out_sig}@1")
                
            match gate_type:
                case "AND" | "NAND":
                    for inp in in_sigs:
                        collapsed_set.add(f"{inp}@1")
                        equivalences.add(f"{inp}@0")
                case "OR" | "NOR":
                    for inp in in_sigs:
                        collapsed_set.add(f"{inp}@0")
                        equivalences.add(f"{inp}@1")
                case _:
                    if gate_type == "NOT":
                        not_gates.append(gate_str)
                    for inp in in_sigs:
                        collapsed_set.add(f"{inp}@0")
                        collapsed_set.add(f"{inp}@1")
                        
        for gate_str in not_gates:
            parsed = self._parse_line(gate_str)
            if parsed:
                out_sig, _, in_sigs = parsed
                for val in ["0", "1"]:
                    fault_str = f"{out_sig}@{val}"
                    inverted_val = str(abs(1 - int(val)))
                    if fault_str in equivalences or fault_str in collapsed_set:
                        for inp in in_sigs:
                            collapsed_set.discard(f"{inp}@{inverted_val}")

        self.collapsed_faults = sorted(list(collapsed_set))

    def _compile_to_tensors(self):
        """
        Phase 5: Matrix Tensor Compilation Strategy.
        Transforms names into fixed index coordinates, structures gate execution types 
        into discrete numerical opcodes, and constructs parallel processing layers.
        """
        self.log(f"Compiling circuit topology to PyTorch device tensor arrays ({self.device})...")
        
        # Build global structural indexing records
        all_net_signals = set(self.primary_inputs) | set(self.primary_outputs) | set(self.gates.keys())
        for idx, sig in enumerate(sorted(list(all_net_signals))):
            self.signal_to_idx[sig] = idx
            self.idx_to_signal[idx] = sig
            
        self.total_signal_count = len(self.signal_to_idx)
        
        # Opcode assignments
        op_map = {"BUFF": 0, "NOT": 1, "AND": 2, "NAND": 3, "OR": 4, "NOR": 5, "XOR": 6, "XNOR": 7}
        
        # Structure the layout using evaluation layer vectors
        self.tensor_layers = []
        for layer in self.layer_groups:
            if not layer: continue
            
            # Find the maximum fan-in across this group to establish stable tensor column widths
            max_inputs = max(len(self.gates[gate]["inputs"]) for gate in layer)
            
            layer_outputs = []
            layer_opcodes = []
            layer_inputs = []
            
            for gate in layer:
                g_info = self.gates[gate]
                layer_outputs.append(self.signal_to_idx[gate])
                layer_opcodes.append(op_map.get(g_info["type"], 0))
                
                # Convert inputs to their absolute signal indices (pad with 0 if fewer inputs exist)
                in_idxs = [self.signal_to_idx[inp] for inp in g_info["inputs"]]
                while len(in_idxs) < max_inputs:
                    in_idxs.append(0) 
                layer_inputs.append(in_idxs)
                
            # Convert raw structural lists into native, compiled PyTorch device tensors
            self.tensor_layers.append({
                "outputs": torch.tensor(layer_outputs, dtype=torch.long, device=self.device),
                "opcodes": torch.tensor(layer_opcodes, dtype=torch.uint8, device=self.device),
                "inputs": torch.tensor(layer_inputs, dtype=torch.long, device=self.device)
            })
            
        self.log(f"Successfully compiled {len(self.tensor_layers)} parallel hardware processing layers.")

    def simulate_vectors(self, input_tensor: torch.Tensor) -> torch.Tensor:
        """
        Executes a highly parallel forward logic simulation across all inputs simultaneously.
        
        :param input_tensor: A Boolean PyTorch tensor of shape (num_test_vectors, num_primary_inputs)
        :return: An evaluation state matrix mapping output values
        """
        num_vectors = input_tensor.shape[0]
        
        # Initialize full circuit simulation state tracking matrix
        # Shape: (num_test_vectors, total_circuit_signals)
        state_tensor = torch.zeros((num_vectors, self.total_signal_count), dtype=torch.bool, device=self.device)
        
        # Map primary input test states into the tracking matrix simultaneously
        pi_indices = [self.signal_to_idx[pi] for pi in self.primary_inputs]
        state_tensor[:, pi_indices] = input_tensor.to(self.device)
        
        # Sequentially step through parallel computation layers
        for layer in self.tensor_layers:
            out_idx = layer["outputs"]
            opcodes = layer["opcodes"]
            in_idx = layer["inputs"]
            
            # Use advanced index gathering to pull the states of all inputs simultaneously
            # Shape: (num_vectors, num_gates_in_layer, fan_in)
            gathered_inputs = state_tensor[:, in_idx]
            
            # Process gate behaviors across all input vectors simultaneously using tensor logic functions
            for i in range(len(out_idx)):
                op = opcodes[i]
                g_out = out_idx[i]
                g_inps = gathered_inputs[:, i, :] # Inputs for this specific gate across all vectors
                
                if op == 0:    # BUFF
                    state_tensor[:, g_out] = g_inps[:, 0]
                elif op == 1:  # NOT
                    state_tensor[:, g_out] = ~g_inps[:, 0]
                elif op == 2:  # AND
                    state_tensor[:, g_out] = torch.all(g_inps, dim=-1)
                elif op == 3:  # NAND
                    state_tensor[:, g_out] = ~torch.all(g_inps, dim=-1)
                elif op == 4:  # OR
                    state_tensor[:, g_out] = torch.any(g_inps, dim=-1)
                elif op == 5:  # NOR
                    state_tensor[:, g_out] = ~torch.any(g_inps, dim=-1)
                elif op == 6:  # XOR
                    state_tensor[:, g_out] = torch.bitwise_xor(g_inps[:, 0], g_inps[:, 1])
                elif op == 7:  # XNOR
                    state_tensor[:, g_out] = ~torch.bitwise_xor(g_inps[:, 0], g_inps[:, 1])

        # Extract values for the primary output nodes
        po_indices = [self.signal_to_idx[po] for po in self.primary_outputs]
        return state_tensor[:, po_indices]

    def _print_summary(self):
        all_possible_faults = len(self.signals) * 2 + len(self.primary_outputs) * 2
        print("*" * 100)
        print(f"Target Compute Device: {self.device.upper()}")
        print(f"# of PIs:              {len(self.primary_inputs)}")
        print(f"# of POs:              {len(self.primary_outputs)}")
        print(f'# of gates:            {len(self.gates)}')
        print(f"# of fanout stems:     {self.fanout_stems}")
        print(f"# of gate inputs:      {self.gate_inputs}")
        print(f"All possible faults:   {all_possible_faults}")
        print(f"Calculated collapsed:  {len(self.collapsed_faults)}")
        print(f"Time taken:            {(time.perf_counter() - self.t0):09.3f} seconds")
        print("*" * 100)

/usr/local/python/3.14.2/lib/python3.14/site-packages/torch/_subclasses/functional_tensor.py:362: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


In [4]:
# =====================================================================
# Verification Execution Block
# =====================================================================
if __name__ == "__main__":
    parser = BenchCircuitParser("./data.nogit/c6288.bench")
    
    # Example: Simulating 10,000 independent input vectors simultaneously on the GPU/CPU
    # num_pi = len(parser.primary_inputs)
    # random_test_patterns = torch.randint(0, 2, (10000, num_pi), dtype=torch.bool)
    
    # Parallel forward simulation pass across all vectors
    # output_results = parser.simulate_vectors(random_test_patterns)
    # print("Simulation Results Matrix Shape:", output_results.shape)
    pass

# 00000.005 - Parsing file: ./data.nogit/c6288.bench
# 00000.135 - Compiling circuit topology to PyTorch device tensor arrays (cpu)...
# 00000.182 - Successfully compiled 217 parallel hardware processing layers.
****************************************************************************************************
Target Compute Device: CPU
# of PIs:              32
# of POs:              32
# of fanout stems:     1456
# of gate inputs:      4800
All possible faults:   12576
Calculated collapsed:  7744
Time taken:            00000.182 seconds
****************************************************************************************************
